---
title: "DEDL_DEFAIR_quick_start" 
subtitle: "This notebook shows how to start working with DEFAIR (Destination Earth Framework for preparing AI-Ready Data)"
author: "Author: Serena Avolio(EUMETSAT/Starion)"
tags: [DEFAIR, EUMETSAT,zarr,AI-Ready-data]
thumbnail: img/EUMETSAT.png 
license: MIT
copyright: "© 2026 EUMETSAT"
---

<!-- Optional: Add a JupyterHub launch link here. If you don’t have one, you can remove this block. -->

<div style="margin: 6px 0;">
  <a href="https://jupyter.central.data.destination-earth.eu/user-redirect/lab/tree/DEFAIR/DEDL_DEFAIR_quick_start.ipynb" target="_blank" style="text-decoration: none;">
    <span class="launch">🚀 Launch in JupyterHub</span>
  </a>
</div>


# DEFAIR (Destination Earth Framework for preparing AI-Ready Data) - Quick Start

### Contents
- **Objective:** This notebook is designed as an introductory guide to DEFAIR and provides a practical, hands-on walkthrough of the fundamental concepts and workflows required to start working with DEFAIR. It is intended for new users who want to understand how DEFAIR can be used to access, process, and manage geospatial and Earth Observation data.
- **Data Sources:** The examples in this notebook use sample datasets hosted in the DestinE demonstration data repository:
  https://s3.central.data.destination-earth.eu/swift/v1/datalake-demo-data/
  These datasets are provided for testing and learning purposes and represent typical Earth Observation products that can be accessed through the DEFAIR.
- **Methods:** The notebook demonstrates the following key operations:
    - *Loading data* - Users will learn how to connect to available data sources and access datasets accessible via the DestinE Data Lake. This step includes reading data and inspecting its structure and metadata.
    - *Trensforming data* - Once loaded, the data can be manipulated and prepared for analysis. Examples of transformations may include filtering, subsetting, reprojection, aggregation, format conversion, or other preprocessing operations commonly required in geospatial workflows.
    - *Writing outputs* - The notebook shows how processed datasets can be exported and stored in supported formats. This enables users to preserve intermediate results, share outputs with other applications, or make them available for subsequent analysis steps.
    - *Provenance tracking* capability  - .

- **Prerequisites:** Before executing the notebook, ensure that the following requirements are met:

    - Running on Insula Code
        - A valid <a href="https://platform.destine.eu/"> Destination Earth (DestinE) </a> user account is required.
        - The "defair" kernel must be selected to run this notebook.

    - Running on DEDL JupyterHub
        - A valid <a href="https://platform.destine.eu/"> Destination Earth (DestinE) </a> user account is required.
        - Access to <a href="https://application.data.destination-earth.eu/"> EDGE Services</a> EDGE Services is required.
        - The "Python (defair)" kernel must be selected to run this notebook.

- **Expected Output:** After successfully completing this notebook, users will be able to:
     - Access and load sample datasets from the DestinE Data Lake.
     - Understand the basic architecture and purpose of DEFAIR services.
    - Perform simple data transformation and preprocessing operations.
    - Save and manage processed outputs in supported formats.
    - Track the provenance of the produced outputs.
    - Gain the basic knowledge required to adapt the demonstrated workflow to their own datasets and use cases.

At the end of the execution, the notebook should generate example outputs derived from the input datasets and demonstrate a complete end-to-end workflow from data ingestion to result generation using DEFAIR service

## Prerequisites

To run this tutorial, the appropriate access to the DestinE platform is needed:
   - To run this notebook on <a href="https://code.insula.destine.eu/"> Insula Code</a> a <a href="https://platform.destine.eu/"> DestinE user account</a> is needed
   - To run this notebook on <a href="https://jupyter.central.data.destination-earth.eu/hub/"> DEDL JupyterHub</a> the <a href="https://application.data.destination-earth.eu/"> access to EDGE Services is needed</a> is needed.

## Imports

In [115]:
import defair 
print("Defair version: "+defair.__version__)

from defair.logging import setup_logging

# Human-readable output
setup_logging(log_level="WARNING")

Defair version: 0.4.0rc2


## Loading data

DEFAIR comes with several built-in data readers that support lazy loading, enabling efficient handling of large datasets.

Use **list_readers()** to load data from local or remote storage. DEFAIR automatically detects the appropriate reader based on file format.

Reader names generally follow a standardized naming convention: 
- the constellation name is followed by an underscore (`_`),
- then the instrument name and processing level are appended together.

The [reader catalogue](https://cloudferro-dedl-staging.readthedocs-hosted.com/en/latest/working_with_ai_in_the_data_lake/defair/reader-catalogue.html) provides for each available reader the product it supports, the data provider, the provider's original collection ID, and the equivalent collection ID available through DEDL HDA.

In [116]:
from pprint import pprint
from inspect import signature
from defair_data.readers import list_readers

readers = list_readers()
print("Available readers:")

pprint(readers)

Available readers:
['era5grib',
 'metop_amsul1',
 'metop_ascszf1b',
 'metop_ascszfr02',
 'metop_ascszo1b',
 'metop_ascszor02',
 'metop_ascszr1b',
 'metop_ascszrr02',
 'metop_avhrr_amv',
 'metop_avhrrl1',
 'metop_edlst',
 'metop_glbsst',
 'metop_gomel1',
 'metop_gomel1r03',
 'metop_hirs_fdr',
 'metop_hirsl1',
 'metop_iasil1c_all',
 'metop_iasisnd02',
 'metop_iasthr011',
 'metop_mhsl1',
 'metop_osi104',
 'metop_osi150a',
 'metop_osi150b',
 'metop_somo12',
 'metop_somo25',
 'msg15nat',
 'mtg_fci_l1c_nc',
 'mtg_fci_l2_amv',
 'mtg_l2_asr',
 'mtg_l2_clm',
 'mtg_l2_gii',
 'mtg_l2_oca',
 'mtg_l2_olr',
 'mtg_li_af',
 'mtg_li_afa',
 'mtg_li_afr',
 'mtg_li_lef',
 'mtg_li_lfl',
 'mtg_li_lgr',
 'sentinel3_aod',
 'sentinel3_frp',
 'sentinel3_ol_1_efr',
 'sentinel3_ol_1_err',
 'sentinel3_ol_2_wfr',
 'sentinel3_ol_2_wrr',
 'sentinel3_sl_1_rbt',
 'sentinel3_sr1_sra',
 'sentinel3_sr1_sra_a',
 'sentinel3_sr1_sra_bs',
 'sentinel3_sr2_wat',
 'sentinel3_wst']


### Loading from S3 - Reader Auto-Detection

Data can be loaded from **different sources**, local or remote storage as well as directly from DEDL HDA.

Use **Dataset.from_source()** to load data. DEFAIR automatically detects the appropriate reader based on file format.

The code below demonstrate how DEFAIR automatically detects the appropriate reader based on file format and load the data from a remote object storage.

In [117]:
from defair_data.core import Dataset
msg_dataset_1 = Dataset.from_source(
    "s3://datalake-demo-data/defair-demo-data/MSG3-SEVI-MSG15-0100-NA-20260626111243.441000000Z-NA.nat",
    source="s3",
    source_kwargs={
        "endpoint_url": "https://s3.central.data.destination-earth.eu",
        "aws_access_key_id": "f27646e0b43e4fc29803eeb8926b9b4a",
        "aws_secret_access_key": "c416629cf9c544e1ab367b6ad8212a1b"
    }
)

2026-09-11 13:53:47 | WARNING  | [25676e63] defair_data.readers.msg15.msg15_native_reader_plugin:_resolve_channels_and_calibrations:647 - HRV is not part of the default MSG channel selection: it is opt-in because it uses its own 1 km grid. Request it explicitly with channels=["HRV", ...] (the output variable is named "HRV" with use_channel_names=True, otherwise "ch12").
2026-09-11 13:53:47 | WARNING  | [25676e63] defair_data.dask_manager:release_client:785 - Release called but ref_count is already 0


Inspecting data structure and metadata.

In [118]:
print(f"Variables: {list(msg_dataset_1.data.data_vars)}")

print(f"Dimensions: {dict(msg_dataset_1.sizes)}")

Variables: ['ch1', 'ch2', 'ch3', 'ch4', 'ch5', 'ch6', 'ch7', 'ch8', 'ch9', 'ch10', 'ch11', 'geostationary']
Dimensions: {'time': 1, 'y': 3712, 'x': 3712}


In [119]:
msg_dataset_1

### Loading from S3 - Explicit Reader Selection and reader-specific options

You can explicitly select a reader when automatic reader detection is unsuccessful or when a specific reader is required.

The following example shows how to inspect the common configuration options and the reader-specific options available for the **msg15nat** reader.

In [120]:
from inspect import signature
from defair.plugin_manager import load_reader

reader = load_reader("msg15nat")

print("COMMON READER OPTIONS:\n---")
pprint(signature(reader.read))
print("\nSPECIFIC READER OPTIONS AND THEIR DEFAULTS VALUES:\n---")
pprint(signature(type(reader)))
#To display additional details, uncomment the following line:
#help(type(reader))

COMMON READER OPTIONS:
---
<Signature (path: str | os.PathLike, source: str | None = None, source_kwargs: dict[str, typing.Any] | None = None, channels: list[str] | None = None, visir_lines_num: int | None = None, lazy_coordinates: bool = True, include_latlon: bool | None = None, include_aux_metadata: bool = False) -> defair_data.core.Dataset>

SPECIFIC READER OPTIONS AND THEIR DEFAULTS VALUES:
---
<Signature (dask_client_kwargs: dict[str, typing.Any] | None = None, chunks: dict[str, int] | str | None = None, calibration: str | collections.abc.Mapping[str, str] = 'radiance', use_channel_names: bool = False)>


---

In the cell below, we use some reader-specific options:

- *calibration*: when set to "auto", solar channels are returned as reflectance and thermal channels as brightness temperature. Calibration can also be specified independently for each channel using a mapping.
- *use_channel_names*: when set to True, variables are named using their EUMETSAT identifiers (for example, HRV or IR_108).
- *channels*: used here to explicitly request the HRV channel, which is provided on the 1 km grid.

We also use some common reader options:

- *reader*: specifies the reader to use.
- *source*: defines the data source.
- *source_kwargs*: provides additional arguments for configuring the source.

In [121]:
# Specify reader explicitly
msg_dataset_2 = Dataset.from_source(
    "s3://datalake-demo-data/defair-demo-data/MSG3-SEVI-MSG15-0100-NA-20260626112743.658000000Z-NA.nat",
    reader="msg15nat",
    source="s3",               # common reader option
    source_kwargs={            # common reader option
        "endpoint_url": "https://s3.central.data.destination-earth.eu",
        "aws_access_key_id": "f27646e0b43e4fc29803eeb8926b9b4a",
        "aws_secret_access_key": "c416629cf9c544e1ab367b6ad8212a1b"
    },
    calibration="auto",        # reader-specific option
    channels=["HRV","IR_108"], # reader-specific option
    use_channel_names=True     # reader-specific option
)

2026-09-11 13:54:30 | WARNING  | [25676e63] defair_data.dask_manager:release_client:785 - Release called but ref_count is already 0


In [122]:
msg_dataset_2

### Loading from DestinE Data Lake HDA 

The DestinE Harmonized Data Access (HDA) source can be accessed through the `hda` data source plugin. 

The HDA source plugin is responsible for authenticated access to assets stored in the DestinE HDA. Asset discovery is performed separately through the HDA STAC catalog. Once a STAC Item has been identified, its assets can be passed to DEFAIR.

Unlike file-system based sources, HDA does not support directory listing or glob expansion. Instead, products must be discovered through the HDA STAC catalog and then accessed using their asset URLs or asset references.

In the example below:
- `source="hda"` explicitly selects the HDA data source plugin.
- Authentication credentials are provided through `source_kwargs`.
- A product URL obtained from the HDA STAC catalog is passed as the input source.
- The reader loads the product transparently, downloading and caching the file when required.
 
This mechanism allows DEFAIR readers to work directly with data stored in the DestinE HDA without requiring manual downloads.

WARNING: downloadLink assets require downloading the entire zip archive to local disk.
This may require significant disk space and processes data only after the full download completes.
It is recommended to use individual assets directly instead of downloadLink
when available.

In [123]:
%pip install --quiet pystac_client


[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: /home/jovyan/defair_venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [124]:
from getpass import getpass
import destinelab as deauth
DESP_USERNAME = input("Please input your DESP username or email: ")
DESP_PASSWORD = getpass("Please input your DESP password: ")

auth = deauth.AuthHandler(DESP_USERNAME, DESP_PASSWORD)
access_token = auth.get_token()
if access_token is not None:
    print("DEDL/DESP Access Token Obtained Successfully")
else:
    print("Failed to Obtain DEDL/DESP Access Token")

auth_headers = {"Authorization": f"Bearer {access_token}"}

Please input your DESP username or email:  eum-dedl-user
Please input your DESP password:  ········


DEDL/DESP Access Token Obtained Successfully


Response code: 200
DEDL/DESP Access Token Obtained Successfully


In [125]:
from pystac_client import Client

catalog = Client.open("https://hda.data.destination-earth.eu/stac/v2", headers=auth_headers)

search = catalog.search(
    collections=["EO.EUM.DAT.MTG.FCI-CLM"],
    datetime="2026-06-26T11:00:00Z/2026-06-26T12:00:00Z",
)

item = next(search.items())

item.assets.keys()

dict_keys(['EOPMetadata.xml', 'W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-2-CLM--FD------NC4E_C_EUMT_20260626110454_L2PF_OPE_20260626105000_20260626110000_N__O_0066_0000.nc', 'W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-2-CLM--FD--QCK-IMAGE---PNG_C_EUMT_20260626110454_L2PF_OPE_20260626105000_20260626110000_N__O_0066_0000.jpg', 'W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-2-CLM--FD--QCK-IMAGE---PNG_C_EUMT_20260626110454_L2PF_OPE_20260626105000_20260626110000_N__O_0066_0000.png', 'manifest.xml', 'downloadLink'])

In [126]:
print(item.assets["W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-2-CLM--FD------NC4E_C_EUMT_20260626110454_L2PF_OPE_20260626105000_20260626110000_N__O_0066_0000.nc"].href)

https://hda-download.central.data.destination-earth.eu/data/eumetsat/EO.EUM.DAT.MTG.FCI-CLM/W_XX-EUMETSAT-Darmstadt%2CIMG%2BSAT%2CMTI1%2BFCI-2-CLM--FD--x-x---x_C_EUMT_20260626110831_L2PF_OPE_20260626105000_20260626110000_N__O_0066_0000/W_XX-EUMETSAT-Darmstadt%2CIMG%2BSAT%2CMTI1%2BFCI-2-CLM--FD------NC4E_C_EUMT_20260626110454_L2PF_OPE_20260626105000_20260626110000_N__O_0066_0000.nc


In [127]:
hda_cube = Dataset.from_source(
    item.assets["W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-2-CLM--FD------NC4E_C_EUMT_20260626110454_L2PF_OPE_20260626105000_20260626110000_N__O_0066_0000.nc"].href,
    reader="mtg_l2_clm",
    source="hda",
    source_kwargs={
        "username": DESP_USERNAME,
        "password": DESP_PASSWORD,
    },
)

hda_cube

### Loading multi-file

By default, DEFAIR creates a single logical dataset and loads array data only when it is actually needed. This lazy-loading approach allows to interact with the dataset as a whole.

In the following example 3 files are loaded in the same dataset:

- W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214154414_IDPFI_OPE_20251214154007_20251214154017_N__O_0095_0001.nc
- W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214155412_IDPFI_OPE_20251214155007_20251214155017_N__O_0096_0001.nc		
- W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214160420_IDPFI_OPE_20251214160007_20251214160017_N__O_0097_0001.nc




In [128]:
cube = Dataset.from_source(
    "s3://datalake-demo-data/defair-demo-data/*MTI1+FCI-1C-RRAD-FDHSI*_0001.nc",
    reader="mtg_fci_l1c_nc",
    source="s3",
    source_kwargs={
        "endpoint_url": "https://s3.central.data.destination-earth.eu",
        "aws_access_key_id": "f27646e0b43e4fc29803eeb8926b9b4a",
        "aws_secret_access_key": "c416629cf9c544e1ab367b6ad8212a1b"
    },
    use_channel_names=True,
    streaming=False
)

print(vars(cube))



{'_data': <xarray.Dataset> Size: 15GB
Dimensions:      (time: 3, y_1km: 11136, x_1km: 11136, y_2km: 5568, x_2km: 5568)
Coordinates:
  * time         (time) datetime64[ns] 24B 2025-12-14T15:40:07 ... 2025-12-14...
    source_file  (time) object 24B 's3://datalake-demo-data/defair-demo-data/...
  * y_1km        (y_1km) float64 89kB -5.568e+06 -5.566e+06 ... 5.568e+06
  * x_1km        (x_1km) float64 89kB 5.568e+06 5.566e+06 ... -5.568e+06
  * y_2km        (y_2km) float64 45kB -5.567e+06 -5.565e+06 ... 5.567e+06
  * x_2km        (x_2km) float64 45kB 5.567e+06 5.565e+06 ... -5.567e+06
    spatial_ref  int64 8B 0
Data variables: (12/16)
    vis_04       (time, y_1km, x_1km) float32 1GB dask.array<chunksize=(1, 556, 5568), meta=np.ndarray>
    vis_05       (time, y_1km, x_1km) float32 1GB dask.array<chunksize=(1, 556, 5568), meta=np.ndarray>
    vis_06       (time, y_1km, x_1km) float32 1GB dask.array<chunksize=(1, 556, 5568), meta=np.ndarray>
    vis_08       (time, y_1km, x_1km) float32 1G

#### Streaming

For very large collections of files, however, loading and combining the entire dataset may require too much memory. In these cases, you can enable streaming mode. Rather than building the complete dataset in memory, DEFAIR processes a limited number of scene groups at a time and writes the results incrementally to a Zarr store.

Important considerations
- Streaming datasets do not expose the .data property because the complete dataset is never assembled in memory.
- To inspect or analyse the final result, reload the generated Zarr store as a new dataset.

In this example, DEFAIR processes the following six scenes up to three scenes at a time

- W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214154414_IDPFI_OPE_20251214154007_20251214154017_N__O_0095_0001.nc	
- W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214155412_IDPFI_OPE_20251214155007_20251214155017_N__O_0096_0001.nc	
- W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214160420_IDPFI_OPE_20251214160007_20251214160017_N__O_0097_0001.nc	
- W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214160434_IDPFI_OPE_20251214160007_20251214160028_N__O_0097_0002.nc		
- W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214160452_IDPFI_OPE_20251214160011_20251214160042_N__O_0097_0003.nc		
- W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214160502_IDPFI_OPE_20251214160017_20251214160056_N__O_0097_0004.nc


on the output object it is possible to apply a transformation using the method [transfom](https://cloudferro-dedl-staging.readthedocs-hosted.com/en/latest/working_with_ai_in_the_data_lake/defair/reference/api/defair_data/index.html#defair_data.Dataset.transform) and append the output directly to a Zarr store using the method [to_file](https://cloudferro-dedl-staging.readthedocs-hosted.com/en/latest/working_with_ai_in_the_data_lake/defair/reference/api/defair_data/core/index.html#defair_data.core.Dataset.to_file). This keeps memory usage predictable even when working with a large number of files.


In [132]:
cube = Dataset.from_source(
    "s3://datalake-demo-data/defair-demo-data/*MTI1+FCI-1C-RRAD-FDHSI*.nc",
    reader="mtg_fci_l1c_nc",
    source="s3",
    source_kwargs={
        "endpoint_url": "https://s3.central.data.destination-earth.eu",
        "aws_access_key_id": "f27646e0b43e4fc29803eeb8926b9b4a",
        "aws_secret_access_key": "c416629cf9c544e1ab367b6ad8212a1b"
    },
    use_channel_names=True,
    streaming=True,
    stream_batch_size=3,
)

print(vars(cube))

#Uncomment the following two lines to apply a transformation and append the processed output to a Zarr store.
#cube = cube.transform("content_filter", include_vars=["vis_06", "ir_105"])
#cube.to_file("output/archive.zarr", writer="zarrv2")

{'_data': None, '_stream_plan': StreamingPlan(file_groups=(('s3://datalake-demo-data/defair-demo-data/W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214154414_IDPFI_OPE_20251214154007_20251214154017_N__O_0095_0001.nc',), ('s3://datalake-demo-data/defair-demo-data/W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214155412_IDPFI_OPE_20251214155007_20251214155017_N__O_0096_0001.nc',), ('s3://datalake-demo-data/defair-demo-data/W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214160420_IDPFI_OPE_20251214160007_20251214160017_N__O_0097_0001.nc', 's3://datalake-demo-data/defair-demo-data/W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214160434_IDPFI_OPE_20251214160007_20251214160028_N__O_0097_0002.nc', 's3://datalake-demo-data/defair-demo-data/W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20251214160452

## Trensforming data

Once loaded, the data can be manipulated and prepared for analysis. Examples of transformations may include filtering, subsetting, reprojection, aggregation, format conversion, or other preprocessing operations commonly required in geospatial workflows.


## Writing outputs 

The notebook shows how processed datasets can be exported and stored in supported formats. This enables users to preserve intermediate results, share outputs with other applications, or make them available for subsequent analysis steps.


## Provenance tracking capability — the history attribute

Every read and transform appends a line to the CF history attribute, so a cube carries a record of how it was made.

In [ ]:
import re
msg_eu = (
    msg_dataset_1.transform("spatial_filter", lat_min=30, lat_max=60, lon_min=-12, lon_max=34)
    .reproject("EPSG:4326", resolution=0.05, resampling="bilinear")
)
for line in msg_eu.data.attrs.get("history", "(none)").splitlines():
    line = re.sub(r"(source=')[^']*[/\\]([^'/\\]+')", r"\1\2", line)   # show file name, not the path
    print(line[:140])